In [2]:
import os
import time

import numpy as np
import scipy.io
import tensorflow as tf
from scipy.interpolate import griddata
from scipy.optimize import minimize
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker

np.random.seed(1234)
tf.random.set_seed(1234)

# Plot style: readable in the thesis, consistent fonts
plt.rcParams.update({
    "font.size": 11,
    "axes.titlesize": 12,
    "axes.labelsize": 11,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "savefig.dpi": 300,
})


# ══════════════════════════════════════════════════════════════════════════════
# PINN Model
# ══════════════════════════════════════════════════════════════════════════════

class PhysicsInformedNN:
    def __init__(self, x, y, t, u, v, layers, learning_rate=1e-3, dtype=tf.float32):
        self.dtype  = dtype
        self.layers = layers

        # ── Data ──────────────────────────────────────────────────────────────
        X = np.concatenate([x, y, t], axis=1).astype(np.float32)

        self.lb = tf.constant(X.min(0, keepdims=True), dtype=self.dtype)
        self.ub = tf.constant(X.max(0, keepdims=True), dtype=self.dtype)

        self.X = tf.convert_to_tensor(X, dtype=self.dtype)
        self.x = tf.convert_to_tensor(X[:, 0:1], dtype=self.dtype)
        self.y = tf.convert_to_tensor(X[:, 1:2], dtype=self.dtype)
        self.t = tf.convert_to_tensor(X[:, 2:3], dtype=self.dtype)

        self.u = tf.convert_to_tensor(u, dtype=self.dtype)
        self.v = tf.convert_to_tensor(v, dtype=self.dtype)

        # ── Network ───────────────────────────────────────────────────────────
        self.net = self._build_network(layers)
        _ = self.net(tf.zeros((1, layers[0]), dtype=self.dtype))  # build weights

        # ── Trainable PDE parameters (start at 0 — no hints) ─────────────────
        self.lambda_1 = tf.Variable(0.0, dtype=self.dtype, trainable=True, name="lambda_1")
        self.lambda_2 = tf.Variable(0.0, dtype=self.dtype, trainable=True, name="lambda_2")

        # ── Optimizer ─────────────────────────────────────────────────────────
        self.optimizer_adam = tf.keras.optimizers.Adam(learning_rate=learning_rate)

        # ── Training history (for the convergence figure) ─────────────────────
        self.history = {"it": [], "loss": [], "l1": [], "l2": []}
        self.n_adam = 0     # number of Adam iterations (position of the switch line)

    # ── Network architecture ──────────────────────────────────────────────────
    def _build_network(self, layers):
        model = tf.keras.Sequential(name="pinn")
        for width in layers[1:-1]:
            model.add(
                tf.keras.layers.Dense(
                    width,
                    activation=tf.nn.tanh,
                    kernel_initializer=tf.keras.initializers.GlorotNormal(),
                    bias_initializer="zeros",
                    dtype=self.dtype,
                )
            )
        model.add(
            tf.keras.layers.Dense(
                layers[-1],
                activation=None,
                kernel_initializer=tf.keras.initializers.GlorotNormal(),
                bias_initializer="zeros",
                dtype=self.dtype,
            )
        )
        return model

    @property
    def train_vars(self):
        return self.net.trainable_variables + [self.lambda_1, self.lambda_2]

    def neural_net(self, X):
        H = 2.0 * (X - self.lb) / (self.ub - self.lb) - 1.0
        return self.net(H)

    @staticmethod
    def _safe_grad(tape, y, x):
        g = tape.gradient(y, x)
        return tf.zeros_like(x) if g is None else g

    def _record(self, it, loss_value):
        self.history["it"].append(it)
        self.history["loss"].append(float(loss_value))
        self.history["l1"].append(float(self.lambda_1))
        self.history["l2"].append(float(self.lambda_2))

    # ── Physics residuals ─────────────────────────────────────────────────────
    def net_NS(self, x, y, t):
        x = tf.convert_to_tensor(x, dtype=self.dtype)
        y = tf.convert_to_tensor(y, dtype=self.dtype)
        t = tf.convert_to_tensor(t, dtype=self.dtype)

        with tf.GradientTape(persistent=True) as tape3:
            tape3.watch([x, y, t])
            with tf.GradientTape(persistent=True) as tape2:
                tape2.watch([x, y, t])
                with tf.GradientTape(persistent=True) as tape1:
                    tape1.watch([x, y, t])
                    X      = tf.concat([x, y, t], axis=1)
                    out    = self.neural_net(X)
                    psi    = out[:, 0:1]
                    p      = out[:, 1:2]

                u   = self._safe_grad(tape1, psi, y)
                v   = -self._safe_grad(tape1, psi, x)
                p_x = self._safe_grad(tape1, p, x)
                p_y = self._safe_grad(tape1, p, y)

            u_t = self._safe_grad(tape2, u, t)
            u_x = self._safe_grad(tape2, u, x)
            u_y = self._safe_grad(tape2, u, y)
            v_t = self._safe_grad(tape2, v, t)
            v_x = self._safe_grad(tape2, v, x)
            v_y = self._safe_grad(tape2, v, y)

        u_xx = self._safe_grad(tape3, u_x, x)
        u_yy = self._safe_grad(tape3, u_y, y)
        v_xx = self._safe_grad(tape3, v_x, x)
        v_yy = self._safe_grad(tape3, v_y, y)

        del tape1, tape2, tape3

        f_u = (u_t
               + self.lambda_1 * (u * u_x + v * u_y)
               + p_x
               - self.lambda_2 * (u_xx + u_yy))

        f_v = (v_t
               + self.lambda_1 * (u * v_x + v * v_y)
               + p_y
               - self.lambda_2 * (v_xx + v_yy))

        return u, v, p, f_u, f_v

    # ── Loss ──────────────────────────────────────────────────────────────────
    def loss(self, x, y, t, u, v):
        u_pred, v_pred, _, f_u_pred, f_v_pred = self.net_NS(x, y, t)
        return (tf.reduce_sum(tf.square(u - u_pred))
                + tf.reduce_sum(tf.square(v - v_pred))
                + tf.reduce_sum(tf.square(f_u_pred))
                + tf.reduce_sum(tf.square(f_v_pred)))

    # ── Adam ──────────────────────────────────────────────────────────────────
    @tf.function
    def train_step_adam(self):
        with tf.GradientTape() as tape:
            loss_value = self.loss(self.x, self.y, self.t, self.u, self.v)
        grads = tape.gradient(loss_value, self.train_vars)
        grads = [tf.zeros_like(v) if g is None else g
                 for g, v in zip(grads, self.train_vars)]
        self.optimizer_adam.apply_gradients(zip(grads, self.train_vars))
        return loss_value

    def train_adam(self, nIter):
        start_time = time.time()
        for it in range(nIter):
            loss_value = self.train_step_adam()
            if it % 10 == 0:
                self._record(it, loss_value)
                elapsed = time.time() - start_time
                print("It: %d, Loss: %.3e, l1: %.6f, l2: %.6f, Time: %.2fs"
                      % (it, float(loss_value),
                         float(self.lambda_1), float(self.lambda_2), elapsed))
                start_time = time.time()
        self.n_adam = nIter

    # ── L-BFGS-B ──────────────────────────────────────────────────────────────
    def _pack_weights(self):
        return np.concatenate(
            [tf.reshape(v, [-1]).numpy() for v in self.train_vars]
        ).astype(np.float64)

    def _unpack_weights(self, w):
        idx = 0
        for var in self.train_vars:
            size = int(np.prod(var.shape))
            var.assign(tf.constant(
                w[idx:idx + size].reshape(var.shape), dtype=self.dtype))
            idx += size

    def _loss_and_grad_lbfgs(self, w):
        self._unpack_weights(w)
        with tf.GradientTape() as tape:
            loss_value = self.loss(self.x, self.y, self.t, self.u, self.v)
        grads = tape.gradient(loss_value, self.train_vars)
        grads = [tf.zeros_like(v) if g is None else g
                 for g, v in zip(grads, self.train_vars)]
        grad_flat = np.concatenate(
            [tf.reshape(g, [-1]).numpy() for g in grads]).astype(np.float64)
        return float(loss_value), grad_flat

    def _callback_lbfgs(self, w):
        self.lbfgs_iter += 1
        self._unpack_weights(w)
        loss_value = float(self.loss(self.x, self.y, self.t, self.u, self.v))
        self._record(self.n_adam + self.lbfgs_iter, loss_value)
        print("L-BFGS-B Iter: %d/%d, Loss: %.3e, l1: %.6f, l2: %.6f"
              % (self.lbfgs_iter, self.lbfgs_maxiter,
                 loss_value, float(self.lambda_1), float(self.lambda_2)))

    def train_lbfgs(self, maxiter=15000, maxfun=15000,
                    maxcor=50, maxls=50, ftol=1.0 * np.finfo(float).eps):
        self.lbfgs_iter    = 0
        self.lbfgs_maxiter = maxiter
        result = minimize(
            fun=self._loss_and_grad_lbfgs,
            x0=self._pack_weights(),
            jac=True,
            method="L-BFGS-B",
            callback=self._callback_lbfgs,
            options={"maxiter": maxiter, "maxfun": maxfun,
                     "maxcor": maxcor,   "maxls":  maxls, "ftol": ftol},
        )
        self._unpack_weights(result.x)
        print("\nL-BFGS-B finished")
        print("  success :", result.success)
        print("  message :", result.message)
        print("  nit     :", result.nit)
        return result

    def train(self, adam_iters=10000, use_lbfgs=True, lbfgs_options=None):
        self.train_adam(adam_iters)
        if use_lbfgs:
            return self.train_lbfgs(**(lbfgs_options or {}))

    def predict(self, x_star, y_star, t_star):
        x_star = tf.convert_to_tensor(x_star, dtype=self.dtype)
        y_star = tf.convert_to_tensor(y_star, dtype=self.dtype)
        t_star = tf.convert_to_tensor(t_star, dtype=self.dtype)
        u, v, p, _, _ = self.net_NS(x_star, y_star, t_star)
        return u.numpy(), v.numpy(), p.numpy()


# ══════════════════════════════════════════════════════════════════════════════
# Plotting helpers
# ══════════════════════════════════════════════════════════════════════════════

def to_grid(X_star, values, Xi, Yi):
    """Interpolate scattered values onto a regular grid.
    Cubic interpolation leaves NaN (white gaps) near the corners of the
    region; these points are filled with nearest-neighbour values."""
    cubic   = griddata(X_star, values, (Xi, Yi), method='cubic')
    nearest = griddata(X_star, values, (Xi, Yi), method='nearest')
    return np.where(np.isnan(cubic), nearest, cubic)


def plot_fields(Xi, Yi, rows, title, out_base):
    """Predicted vs exact fields, one row per quantity, shared colour scale per row."""
    width  = Xi.max() - Xi.min()
    height = Yi.max() - Yi.min()
    panel_w = 5.0
    panel_h = panel_w * height / width            # keep true aspect ratio
    fig, axes = plt.subplots(len(rows), 2,
                             figsize=(2 * panel_w + 1.2, len(rows) * (panel_h + 0.9) + 0.6),
                             constrained_layout=True)
    fig.suptitle(title)

    for r, (name, pred_field, exact_field, cmap) in enumerate(rows):
        vmin = min(np.min(pred_field), np.min(exact_field))
        vmax = max(np.max(pred_field), np.max(exact_field))
        levels = np.linspace(vmin, vmax, 51)       # same colours in both panels

        for c, (field, label) in enumerate([(pred_field, "predicted"),
                                            (exact_field, "exact")]):
            ax = axes[r, c]
            im = ax.contourf(Xi, Yi, field, levels=levels, cmap=cmap, extend="both")
            ax.set_aspect("equal")
            ax.set_title(f"${name}$ {label}")
            ax.set_xlabel("$x$")
            ax.set_ylabel("$y$")
        cbar = fig.colorbar(im, ax=axes[r, :], shrink=0.9, pad=0.02)
        cbar.locator = matplotlib.ticker.MaxNLocator(nbins=6)   # round tick values
        cbar.update_ticks()

    fig.savefig(out_base + ".png", bbox_inches="tight", pad_inches=0.1)
    fig.savefig(out_base + ".pdf", bbox_inches="tight", pad_inches=0.1)
    plt.close(fig)
    print(f"  Figure saved: {out_base}.png / .pdf")


def plot_fields_smooth(Xi, Yi, rows, title, out_base):
    """Same layout as plot_fields, but drawn as a continuous colour map
    (Gouraud shading) instead of filled contour bands, so there are no
    visible level lines."""
    width  = Xi.max() - Xi.min()
    height = Yi.max() - Yi.min()
    panel_w = 5.0
    panel_h = panel_w * height / width
    fig, axes = plt.subplots(len(rows), 2,
                             figsize=(2 * panel_w + 1.2, len(rows) * (panel_h + 0.9) + 0.6),
                             constrained_layout=True)
    fig.suptitle(title)

    for r, (name, pred_field, exact_field, cmap) in enumerate(rows):
        vmin = min(np.min(pred_field), np.min(exact_field))
        vmax = max(np.max(pred_field), np.max(exact_field))

        for c, (field, label) in enumerate([(pred_field, "predicted"),
                                            (exact_field, "exact")]):
            ax = axes[r, c]
            im = ax.pcolormesh(Xi, Yi, field, cmap=cmap, vmin=vmin, vmax=vmax,
                               shading="gouraud", rasterized=True)
            ax.set_aspect("equal")
            ax.set_title(f"${name}$ {label}")
            ax.set_xlabel("$x$")
            ax.set_ylabel("$y$")
        cbar = fig.colorbar(im, ax=axes[r, :], shrink=0.9, pad=0.02)
        cbar.locator = matplotlib.ticker.MaxNLocator(nbins=6)
        cbar.update_ticks()
        cbar.solids.set_rasterized(True)
        cbar.solids.set_edgecolor("face")

    fig.savefig(out_base + ".png", bbox_inches="tight", pad_inches=0.1)
    fig.savefig(out_base + ".pdf", bbox_inches="tight", pad_inches=0.1, dpi=300)
    plt.close(fig)
    print(f"  Figure saved: {out_base}.png / .pdf")


def plot_history(model, l2_true, title, out_base):
    """Loss and identified lambda_1, lambda_2 during training."""
    it   = np.array(model.history["it"])
    loss = np.array(model.history["loss"])
    l1   = np.array(model.history["l1"])
    l2   = np.array(model.history["l2"])

    fig, axes = plt.subplots(1, 3, figsize=(14, 4), constrained_layout=True)
    fig.suptitle(title)

    axes[0].semilogy(it, loss, lw=1.2)
    axes[0].set_ylabel("loss")
    axes[1].plot(it, l1, lw=1.2, label=r"identified $\lambda_1$")
    axes[1].axhline(1.0, color="k", ls="--", lw=1, label="true value")
    axes[1].set_ylabel(r"$\lambda_1$")
    axes[2].plot(it, l2, lw=1.2, label=r"identified $\lambda_2$")
    axes[2].axhline(l2_true, color="k", ls="--", lw=1, label="true value")
    axes[2].set_ylabel(r"$\lambda_2$")

    # keep the true-value line inside the plot with some margin
    for ax, vals, true in [(axes[1], l1, 1.0), (axes[2], l2, l2_true)]:
        lo, hi = min(vals.min(), true), max(vals.max(), true)
        pad = 0.08 * (hi - lo if hi > lo else abs(true))
        ax.set_ylim(lo - pad, hi + pad)

    for ax in axes:
        ax.set_xlabel("iteration")
        if model.n_adam < it.max():
            ax.axvline(model.n_adam, color="gray", ls=":", lw=1)
        ax.grid(alpha=0.3)
    axes[1].legend(loc="lower right")
    axes[2].legend(loc="lower right")
    # mark where L-BFGS-B starts
    axes[0].text(model.n_adam, axes[0].get_ylim()[1], " L-BFGS-B",
                 va="top", ha="left", fontsize=9, color="gray")

    fig.savefig(out_base + ".png", bbox_inches="tight", pad_inches=0.1)
    fig.savefig(out_base + ".pdf", bbox_inches="tight", pad_inches=0.1)
    plt.close(fig)
    print(f"  Figure saved: {out_base}.png / .pdf")


# ══════════════════════════════════════════════════════════════════════════════
# Main
# ══════════════════════════════════════════════════════════════════════════════

if __name__ == "__main__":
    # ── Config ────────────────────────────────────────────────────────────────
    MAT_FILE     = "cylinder_nektar_new_far.mat"
    RE           = 100                 
    LAMBDA2_TRUE = 1.0 / RE            
    N_train      = 5000
    layers       = [3, 20, 20, 20, 20, 20, 20, 20, 20, 2]
    ADAM_ITERS        = 30000
    LBFGS_OPTIONS     = {"maxiter": 15000, "maxfun": 50000}
    NOISY_ADAM_ITERS  = 200000          
    NOISE             = 0.01

    # output names follow the data set, e.g. "cylinder_nektar_mid"
    tag     = os.path.splitext(os.path.basename(MAT_FILE))[0]
    out_dir = f"results_{tag}"
    os.makedirs(out_dir, exist_ok=True)

    print("=" * 60)
    print(f"  Dataset  : {MAT_FILE}")
    print(f"  Re       : {RE}   (true lambda_2 = {LAMBDA2_TRUE})")
    print(f"  N_train  : {N_train}")
    print(f"  Layers   : {layers}")
    print("=" * 60)

    # ── Load data ─────────────────────────────────────────────────────────────
    data = scipy.io.loadmat(MAT_FILE)

    # cylinder_nektar_mid.mat keys (Nektar++ extraction, unstructured points)
    # x_star, y_star : N x 1,  t_star : T x 1,  U_star, V_star, P_star : N x T
    U_star = data['U_star']             # N x T
    V_star = data['V_star']             # N x T
    P_star = data['P_star']             # N x T
    t_star = data['t_star'].flatten()   # T,
    x_star = data['x_star'].flatten()   # N,
    y_star = data['y_star'].flatten()   # N,

    N = x_star.shape[0]
    T = t_star.shape[0]

    print(f"\n  N points    : {N}")
    print(f"  T snapshots : {T}")
    print(f"  x range     : [{x_star.min():.3f}, {x_star.max():.3f}]")
    print(f"  y range     : [{y_star.min():.3f}, {y_star.max():.3f}]")
    print(f"  t range     : [{t_star.min():.3f}, {t_star.max():.3f}]\n")

    # ── Rearrange into flat (NT x 1) arrays ──────────────────────────────────
    XX = np.tile(x_star[:, None], (1, T))   # N x T
    YY = np.tile(y_star[:, None], (1, T))   # N x T
    TT = np.tile(t_star[None, :], (N, 1))   # N x T

    x = XX.flatten()[:, None]   # NT x 1
    y = YY.flatten()[:, None]   # NT x 1
    t = TT.flatten()[:, None]   # NT x 1
    u = U_star.flatten()[:, None]   # NT x 1
    v = V_star.flatten()[:, None]   # NT x 1
    p = P_star.flatten()[:, None]   # NT x 1

    # ── Random training subset ────────────────────────────────────────────────
    idx     = np.random.choice(N * T, N_train, replace=False)
    x_train = x[idx, :]
    y_train = y[idx, :]
    t_train = t[idx, :]
    u_train = u[idx, :]
    v_train = v[idx, :]

    ######################################################################
    ########################### Clean Data ###############################
    ######################################################################
    model = PhysicsInformedNN(x_train, y_train, t_train,
                              u_train, v_train, layers)
    model.train(adam_iters=ADAM_ITERS, use_lbfgs=True, lbfgs_options=LBFGS_OPTIONS)

    lambda_1_value = float(model.lambda_1)
    lambda_2_value = float(model.lambda_2)

    # ── Test at one snapshot ──────────────────────────────────────────────────
    snap = min(100, T - 1)

    x_test = x_star[:, None]                                  # N x 1
    y_test = y_star[:, None]                                  # N x 1
    t_test = np.full((N, 1), t_star[snap], dtype=np.float32)  # N x 1

    u_star_test = U_star[:, snap:snap + 1]   # N x 1
    v_star_test = V_star[:, snap:snap + 1]   # N x 1
    p_star_test = P_star[:, snap:snap + 1]   # N x 1

    u_pred, v_pred, p_pred = model.predict(x_test, y_test, t_test)

    # Pressure is only determined up to a constant: shift the predicted
    # pressure so that its mean equals the mean of the reference pressure.
    p_pred = p_pred - p_pred.mean() + p_star_test.mean()

    # ── Errors ────────────────────────────────────────────────────────────────
    error_u = np.linalg.norm(u_star_test - u_pred, 2) / np.linalg.norm(u_star_test, 2)
    error_v = np.linalg.norm(v_star_test - v_pred, 2) / np.linalg.norm(v_star_test, 2)
    error_p = np.linalg.norm(p_star_test - p_pred, 2) / np.linalg.norm(p_star_test, 2)

    error_lambda_1 = np.abs(lambda_1_value - 1.0) * 100
    error_lambda_2 = np.abs(lambda_2_value - LAMBDA2_TRUE) / LAMBDA2_TRUE * 100

    # ── Figures for the clean run ─────────────────────────────────────────────
    X_star = np.hstack([x_star[:, None], y_star[:, None]])   # N x 2
    lb = X_star.min(0)
    ub = X_star.max(0)
    nn = 300
    xi = np.linspace(lb[0], ub[0], nn)
    yi = np.linspace(lb[1], ub[1], nn)
    Xi, Yi = np.meshgrid(xi, yi)

    rows = [
        ("u", to_grid(X_star, u_pred.flatten(), Xi, Yi),
              to_grid(X_star, u_star_test.flatten(), Xi, Yi), "RdBu_r"),
        ("v", to_grid(X_star, v_pred.flatten(), Xi, Yi),
              to_grid(X_star, v_star_test.flatten(), Xi, Yi), "RdBu_r"),
        ("p", to_grid(X_star, p_pred.flatten(), Xi, Yi),
              to_grid(X_star, p_star_test.flatten(), Xi, Yi), "viridis"),
    ]
    plot_fields(Xi, Yi, rows,
                title=(rf"$Re={RE}$, {tag}, $t={t_star[snap]:.2f}$:  "
                       rf"$\lambda_1={lambda_1_value:.4f}$, $\lambda_2={lambda_2_value:.5f}$"),
                out_base=os.path.join(out_dir, f"fields_{tag}"))
    plot_fields_smooth(Xi, Yi, rows,
                title=(rf"$Re={RE}$, {tag}, $t={t_star[snap]:.2f}$:  "
                       rf"$\lambda_1={lambda_1_value:.4f}$, $\lambda_2={lambda_2_value:.5f}$"),
                out_base=os.path.join(out_dir, f"fields_smooth_{tag}"))
    plot_history(model, LAMBDA2_TRUE,
                 title=rf"Training history, $Re={RE}$, {tag}",
                 out_base=os.path.join(out_dir, f"history_{tag}"))

    ######################################################################
    ########################### Noisy Data ###############################
    ######################################################################
    u_train_noisy = u_train + NOISE * np.std(u_train) * np.random.randn(*u_train.shape)
    v_train_noisy = v_train + NOISE * np.std(v_train) * np.random.randn(*v_train.shape)

    # Training with Adam only (no L-BFGS-B)
    model_noisy = PhysicsInformedNN(x_train, y_train, t_train,
                                    u_train_noisy, v_train_noisy, layers)
    model_noisy.train(adam_iters=NOISY_ADAM_ITERS, use_lbfgs=False)

    lambda_1_value_noisy = float(model_noisy.lambda_1)
    lambda_2_value_noisy = float(model_noisy.lambda_2)

    error_lambda_1_noisy = np.abs(lambda_1_value_noisy - 1.0) * 100
    error_lambda_2_noisy = np.abs(lambda_2_value_noisy - LAMBDA2_TRUE) / LAMBDA2_TRUE * 100

    ######################################################################
    ############################# Results ################################
    ######################################################################
    lines = [
        "=" * 60,
        f"  RESULTS  ({MAT_FILE}, Re = {RE})",
        "=" * 60,
        "  Clean data (Adam + L-BFGS-B):",
        f"    lambda_1 = {lambda_1_value:.6f}   (true = 1.0,  error = {error_lambda_1:.4f}%)",
        f"    lambda_2 = {lambda_2_value:.6f}   (true = {LAMBDA2_TRUE}, error = {error_lambda_2:.4f}%)",
        f"    rel. L2 error u : {error_u:.4e}",
        f"    rel. L2 error v : {error_v:.4e}",
        f"    rel. L2 error p : {error_p:.4e}   (after mean shift)",
        f"  Noisy data, {NOISE*100:.0f}% noise (Adam only, {NOISY_ADAM_ITERS} iterations):",
        f"    lambda_1 = {lambda_1_value_noisy:.6f}   (error = {error_lambda_1_noisy:.4f}%)",
        f"    lambda_2 = {lambda_2_value_noisy:.6f}   (error = {error_lambda_2_noisy:.4f}%)",
        "=" * 60,
    ]
    print("\n" + "\n".join(lines))
    with open(os.path.join(out_dir, f"results_{tag}.txt"), "w") as fh:
        fh.write("\n".join(lines) + "\n")
    print(f"  Results saved: {out_dir}/results_{tag}.txt")

  Dataset  : cylinder_nektar_new_far.mat
  Re       : 100   (true lambda_2 = 0.01)
  N_train  : 5000
  Layers   : [3, 20, 20, 20, 20, 20, 20, 20, 20, 2]

  N points    : 1567
  T snapshots : 400
  x range     : [15.006, 19.994]
  y range     : [-2.498, -0.502]
  t range     : [0.000, 39.900]

It: 0, Loss: 5.247e+03, l1: 0.001000, l2: 0.001000, Time: 10.38s
It: 10, Loss: 1.718e+03, l1: 0.009318, l2: 0.006150, Time: 0.31s
It: 20, Loss: 7.773e+02, l1: 0.008634, l2: 0.008419, Time: 0.29s
It: 30, Loss: 5.340e+02, l1: -0.001924, l2: -0.000850, Time: 0.29s
It: 40, Loss: 4.611e+02, l1: -0.003310, l2: -0.008793, Time: 0.29s
It: 50, Loss: 3.924e+02, l1: 0.000720, l2: -0.010445, Time: 0.29s
It: 60, Loss: 3.432e+02, l1: 0.004618, l2: -0.005412, Time: 0.29s
It: 70, Loss: 3.116e+02, l1: 0.010773, l2: -0.002575, Time: 0.29s
It: 80, Loss: 2.882e+02, l1: 0.017656, l2: -0.001625, Time: 0.29s
It: 90, Loss: 2.713e+02, l1: 0.024222, l2: -0.000934, Time: 0.29s
It: 100, Loss: 2.594e+02, l1: 0.030091, l2: -0.